In [6]:
# Imports and constants
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import joblib

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent.parent
EXP_DIR = PROJECT_ROOT / 'experiments' / 'step0_baselines'
MODELS_DIR = EXP_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Experiment dir:', EXP_DIR)

Project root: /Users/kirtan/Projects /NNDL
Experiment dir: /Users/kirtan/Projects /NNDL/experiments/step0_baselines


In [7]:
# Load data
X_train = np.load(PROJECT_ROOT / 'step3_X_train.npy').astype(np.float32)
X_test = np.load(PROJECT_ROOT / 'step3_X_test.npy').astype(np.float32)
y_train = np.load(PROJECT_ROOT / 'step3_y_train.npy')
y_test = np.load(PROJECT_ROOT / 'step3_y_test.npy')
print('Shapes: ', X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Target labeled subset (optional)
X_target_filtered = None
y_target_filtered = None
labels_csv = PROJECT_ROOT / 'improvements' / '1_label_validation' / 'label_validation_output' / 'gse126030_reclustered_labels.csv'
if labels_csv.exists():
    df = pd.read_csv(labels_csv)
    try:
        target_path = PROJECT_ROOT / 'improvements' / '2_batch_correction' / 'batch_correction_output' / 'gse126030_corrected_coral.npy'
        if target_path.exists():
            X_target = np.load(target_path).astype(np.float32)
            if 'confidence' in df.columns and 'new_class' in df.columns:
                mask = (df['confidence'] >= 0.55) & (df['new_class'] != 'Uncertain')
                label_map = json.load(open(PROJECT_ROOT / 'step3_label_mapping.json'))
                class_names = [label_map[str(i)] for i in range(len(label_map))]
                X_target_filtered = X_target[mask.values]
                y_target_filtered = np.array([class_names.index(n) for n in df.loc[mask, 'new_class']], dtype=np.int64)
                print('Loaded labeled target subset:', X_target_filtered.shape)
    except Exception as e:
        print('Could not load target labeled subset:', e)

Shapes:  (6824, 3000) (1706, 3000) (6824,) (1706,)


In [8]:
# Run Logistic Regression
print('Training Logistic Regression...')
clf_lr = LogisticRegression(max_iter=2000, random_state=SEED, n_jobs=-1)
clf_lr.fit(X_train, y_train)

pred_test = clf_lr.predict(X_test)
print('Source macro-F1:', f1_score(y_test, pred_test, average='macro'))
print('Source accuracy:', accuracy_score(y_test, pred_test))

if X_target_filtered is not None and y_target_filtered is not None:
    pred_target = clf_lr.predict(X_target_filtered)
    print('Target macro-F1:', f1_score(y_target_filtered, pred_target, average='macro'))
    print('Target accuracy:', accuracy_score(y_target_filtered, pred_target))
joblib.dump(clf_lr, MODELS_DIR / 'logreg.joblib')

Training Logistic Regression...


/Users/kirtan/Projects /NNDL/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Source macro-F1: 0.885286214033179
Source accuracy: 0.8968347010550997


['/Users/kirtan/Projects /NNDL/experiments/step0_baselines/models/logreg.joblib']

In [9]:
# Run Random Forest
print('\nTraining Random Forest...')
clf_rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
clf_rf.fit(X_train, y_train)

pred_test = clf_rf.predict(X_test)
print('Source macro-F1:', f1_score(y_test, pred_test, average='macro'))
print('Source accuracy:', accuracy_score(y_test, pred_test))

if X_target_filtered is not None and y_target_filtered is not None:
    pred_target = clf_rf.predict(X_target_filtered)
    print('Target macro-F1:', f1_score(y_target_filtered, pred_target, average='macro'))
    print('Target accuracy:', accuracy_score(y_target_filtered, pred_target))
joblib.dump(clf_rf, MODELS_DIR / 'random_forest.joblib')


Training Random Forest...
Source macro-F1: 0.861778771996892
Source accuracy: 0.8751465416178195


['/Users/kirtan/Projects /NNDL/experiments/step0_baselines/models/random_forest.joblib']

In [10]:
# Save results
results = []
for name, clf in [('LogisticRegression', clf_lr), ('RandomForest', clf_rf)]:
    pred = clf.predict(X_test)
    results.append({'model':name,'domain':'source','f1_macro':float(f1_score(y_test,pred,average='macro')),'accuracy':float(accuracy_score(y_test,pred))})
    if X_target_filtered is not None and y_target_filtered is not None:
        pred_t = clf.predict(X_target_filtered)
        results.append({'model':name,'domain':'target','f1_macro':float(f1_score(y_target_filtered,pred_t,average='macro')),'accuracy':float(accuracy_score(y_target_filtered,pred_t))})

pd.DataFrame(results).to_csv(EXP_DIR / 'results.csv', index=False)
print('Saved results to', EXP_DIR / 'results.csv')

Saved results to /Users/kirtan/Projects /NNDL/experiments/step0_baselines/results.csv


## Next steps
- Run CORAL alignment and re-run baselines
- Implement DANN training script in `experiments/step1_dann/`